# 01 — Data Extraction

**Sector:** Aviation / Transportation 
**Dataset:** US Flight Delays 2015 
**Objective:** Extract and load all raw datasets, perform initial quality assessment, and document data structure.

---

## 1.1 Install Dependencies & Import Libraries

In [1]:
# Install required packages (uncomment if running for the first time)
# !pip install pandas numpy matplotlib seaborn scipy statsmodels

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries loaded successfully.
Pandas version: 3.0.2
NumPy version: 2.4.4


## 1.2 Define Data Paths (Auto-Detect)

In [2]:
# Auto-detect project root and data paths
# Works whether notebook is run from /notebooks/, project root, or Google Colab

def find_data_path():
    """Auto-detect the location of raw CSV files."""
    candidates = [
        '../data/raw/',          # Running from notebooks/ folder
        'data/raw/',             # Running from project root
        './',                    # CSVs in current directory
        '../',                   # One level up
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'airlines.csv')):
            print(f"✅ Data found at: {os.path.abspath(path)}")
            return path
    raise FileNotFoundError("Could not find airlines.csv. Please check your file locations.")

RAW_DATA_PATH = find_data_path()

# Set processed output path relative to raw path
if RAW_DATA_PATH == '../data/raw/':
    PROCESSED_DATA_PATH = '../data/processed/'
elif RAW_DATA_PATH == 'data/raw/':
    PROCESSED_DATA_PATH = 'data/processed/'
else:
    PROCESSED_DATA_PATH = os.path.join(RAW_DATA_PATH, '..', 'data', 'processed') + '/'

airlines_path = os.path.join(RAW_DATA_PATH, 'airlines.csv')
airports_path = os.path.join(RAW_DATA_PATH, 'airports.csv')
flights_path = os.path.join(RAW_DATA_PATH, 'flights.csv')

# Verify files exist
for path in [airlines_path, airports_path, flights_path]:
    exists = os.path.exists(path)
    size_mb = os.path.getsize(path) / (1024 * 1024) if exists else 0
    print(f"{'✅' if exists else '❌'} {os.path.basename(path):25s} | Exists: {exists} | Size: {size_mb:.2f} MB")

✅ Data found at: /Users/yashraghubanshi/Downloads/dva/data/raw
✅ airlines.csv              | Exists: True | Size: 0.00 MB
✅ airports.csv              | Exists: True | Size: 0.02 MB
✅ flights.csv               | Exists: True | Size: 564.96 MB


## 1.3 Load Airlines Dataset

In [3]:
# Load airlines reference table
df_airlines = pd.read_csv(airlines_path)

print(f"Shape: {df_airlines.shape}")
print(f"\nColumn Dtypes:")
print(df_airlines.dtypes)
print(f"\nMissing Values:")
print(df_airlines.isnull().sum())
print(f"\nDuplicate Rows: {df_airlines.duplicated().sum()}")
print(f"\n--- Preview ---")
df_airlines.head(15)

Shape: (14, 2)

Column Dtypes:
IATA_CODE    str
AIRLINE      str
dtype: object

Missing Values:
IATA_CODE    0
AIRLINE      0
dtype: int64

Duplicate Rows: 0

--- Preview ---


,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.
3,F9,Frontier Airlines Inc.
4,B6,JetBlue Airways
5,OO,Skywest Airlines Inc.
6,AS,Alaska Airlines Inc.
7,NK,Spirit Air Lines
8,WN,Southwest Airlines Co.
9,DL,Delta Air Lines Inc.


## 1.4 Load Airports Dataset

In [4]:
# Load airports reference table
df_airports = pd.read_csv(airports_path)

print(f"Shape: {df_airports.shape}")
print(f"\nColumn Dtypes:")
print(df_airports.dtypes)
print(f"\nMissing Values:")
print(df_airports.isnull().sum())
print(f"\nDuplicate Rows: {df_airports.duplicated().sum()}")
print(f"\nUnique States: {df_airports['STATE'].nunique()}")
print(f"\n--- Preview ---")
df_airports.head(10)

Shape: (322, 7)

Column Dtypes:
IATA_CODE        str
AIRPORT          str
CITY             str
STATE            str
COUNTRY          str
LATITUDE     float64
LONGITUDE    float64
dtype: object

Missing Values:
IATA_CODE    0
AIRPORT      0
CITY         0
STATE        0
COUNTRY      0
LATITUDE     3
LONGITUDE    3
dtype: int64

Duplicate Rows: 0

Unique States: 54

--- Preview ---


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447
5,ACK,Nantucket Memorial Airport,Nantucket,MA,USA,41.25305,-70.06018
6,ACT,Waco Regional Airport,Waco,TX,USA,31.61129,-97.23052
7,ACV,Arcata Airport,Arcata/Eureka,CA,USA,40.97812,-124.10862
8,ACY,Atlantic City International Airport,Atlantic City,NJ,USA,39.45758,-74.57717
9,ADK,Adak Airport,Adak,AK,USA,51.87796,-176.64603


## 1.5 Load Flights Dataset

In [5]:
# Load flights dataset — this is the main dataset (~5.8M rows)
print("Loading flights dataset... (this may take a moment due to 5.8M+ rows)")
df_flights = pd.read_csv(flights_path, low_memory=False)
print(f"✅ Flights dataset loaded successfully.")

print(f"\nShape: {df_flights.shape}")
print(f"Memory Usage: {df_flights.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

Loading flights dataset... (this may take a moment due to 5.8M+ rows)
✅ Flights dataset loaded successfully.

Shape: (5819079, 31)
Memory Usage: 2500.34 MB


In [6]:
# Column data types
print("Column Data Types:")
print("=" * 50)
for col in df_flights.columns:
    print(f"  {col:30s} | {str(df_flights[col].dtype):10s} | Non-null: {df_flights[col].notna().sum():>10,}")

Column Data Types:
  YEAR                           | int64      | Non-null:  5,819,079
  MONTH                          | int64      | Non-null:  5,819,079
  DAY                            | int64      | Non-null:  5,819,079
  DAY_OF_WEEK                    | int64      | Non-null:  5,819,079
  AIRLINE                        | str        | Non-null:  5,819,079
  FLIGHT_NUMBER                  | int64      | Non-null:  5,819,079
  TAIL_NUMBER                    | str        | Non-null:  5,804,358
  ORIGIN_AIRPORT                 | str        | Non-null:  5,819,079
  DESTINATION_AIRPORT            | str        | Non-null:  5,819,079
  SCHEDULED_DEPARTURE            | int64      | Non-null:  5,819,079
  DEPARTURE_TIME                 | float64    | Non-null:  5,732,926
  DEPARTURE_DELAY                | float64    | Non-null:  5,732,926
  TAXI_OUT                       | float64    | Non-null:  5,730,032
  WHEELS_OFF                     | float64    | Non-null:  5,730,032
  SCHEDULED_TIM

In [7]:
# Preview flights data
print("--- Flights Preview ---")
df_flights.head(10)

--- Flights Preview ---


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
5,2015,1,1,4,DL,806,N3730B,SFO,MSP,25,...,610.0,8.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
6,2015,1,1,4,NK,612,N635NK,LAS,MSP,25,...,509.0,-17.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
7,2015,1,1,4,US,2013,N584UW,LAX,CLT,30,...,753.0,-10.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
8,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,30,...,532.0,-13.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
9,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,656.0,-15.0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## 1.6 Initial Quality Assessment

In [8]:
# Missing values analysis for flights
missing = df_flights.isnull().sum()
missing_pct = (missing / len(df_flights) * 100).round(2)

quality_report = pd.DataFrame({
    'Column': df_flights.columns,
    'Dtype': df_flights.dtypes.values,
    'Non-Null Count': df_flights.notna().sum().values,
    'Missing Count': missing.values,
    'Missing %': missing_pct.values,
    'Unique Values': df_flights.nunique().values
})

print("=" * 90)
print("FLIGHTS DATASET — DATA QUALITY REPORT")
print("=" * 90)
quality_report

FLIGHTS DATASET — DATA QUALITY REPORT


,Column,Dtype,Non-Null Count,Missing Count,Missing %,Unique Values
0,YEAR,int64,5819079,0,0.00,1
1,MONTH,int64,5819079,0,0.00,12
2,DAY,int64,5819079,0,0.00,31
3,DAY_OF_WEEK,int64,5819079,0,0.00,7
4,AIRLINE,str,5819079,0,0.00,14
5,FLIGHT_NUMBER,int64,5819079,0,0.00,6952
6,TAIL_NUMBER,str,5804358,14721,0.25,4897
7,ORIGIN_AIRPORT,str,5819079,0,0.00,628
8,DESTINATION_AIRPORT,str,5819079,0,0.00,629
9,SCHEDULED_DEPARTURE,int64,5819079,0,0.00,1321


In [9]:
# Columns with missing values
print("\nColumns with Missing Values (sorted by % missing):")
print("=" * 60)
cols_with_missing = quality_report[quality_report['Missing %'] > 0].sort_values('Missing %', ascending=False)
for _, row in cols_with_missing.iterrows():
    print(f"  {row['Column']:30s} | {row['Missing Count']:>10,} missing | {row['Missing %']:>6.2f}%")


Columns with Missing Values (sorted by % missing):
  CANCELLATION_REASON            |  5,729,195 missing |  98.46%
  WEATHER_DELAY                  |  4,755,640 missing |  81.72%
  LATE_AIRCRAFT_DELAY            |  4,755,640 missing |  81.72%
  AIRLINE_DELAY                  |  4,755,640 missing |  81.72%
  SECURITY_DELAY                 |  4,755,640 missing |  81.72%
  AIR_SYSTEM_DELAY               |  4,755,640 missing |  81.72%
  ARRIVAL_DELAY                  |    105,071 missing |   1.81%
  ELAPSED_TIME                   |    105,071 missing |   1.81%
  AIR_TIME                       |    105,071 missing |   1.81%
  TAXI_IN                        |     92,513 missing |   1.59%
  ARRIVAL_TIME                   |     92,513 missing |   1.59%
  WHEELS_ON                      |     92,513 missing |   1.59%
  WHEELS_OFF                     |     89,047 missing |   1.53%
  TAXI_OUT                       |     89,047 missing |   1.53%
  DEPARTURE_TIME                 |     86,153 missin

## 1.7 Summary Statistics

In [10]:
# Numerical summary statistics
print("Numerical Summary Statistics:")
df_flights.describe().round(2)

Numerical Summary Statistics:


,YEAR,MONTH,DAY,DAY_OF_WEEK,FLIGHT_NUMBER,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,...,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
count,5819079.0,5819079.00,5819079.00,5819079.00,5819079.00,5819079.00,5732926.00,5732926.00,5730032.00,5730032.00,...,5819079.00,5726566.00,5714008.00,5819079.00,5819079.00,1063439.00,1063439.00,1063439.00,1063439.00,1063439.00
mean,2015.0,6.52,15.70,3.93,2173.09,1329.60,1335.20,9.37,16.07,1357.17,...,1493.81,1476.49,4.41,0.00,0.02,13.48,0.08,18.97,23.47,2.92
std,0.0,3.41,8.78,1.99,1757.06,483.75,496.42,37.08,8.90,498.01,...,507.16,526.32,39.27,0.05,0.12,28.00,2.14,48.16,43.20,20.43
min,2015.0,1.00,1.00,1.00,1.00,1.00,1.00,-82.00,1.00,1.00,...,1.00,1.00,-87.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,2015.0,4.00,8.00,2.00,730.00,917.00,921.00,-5.00,11.00,935.00,...,1110.00,1059.00,-13.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,2015.0,7.00,16.00,4.00,1690.00,1325.00,1330.00,-2.00,14.00,1343.00,...,1520.00,1512.00,-5.00,0.00,0.00,2.00,0.00,2.00,3.00,0.00
75%,2015.0,9.00,23.00,6.00,3230.00,1730.00,1740.00,7.00,19.00,1754.00,...,1918.00,1917.00,8.00,0.00,0.00,18.00,0.00,19.00,29.00,0.00
max,2015.0,12.00,31.00,7.00,9855.00,2359.00,2400.00,1988.00,225.00,2400.00,...,2400.00,2400.00,1971.00,1.00,1.00,1134.00,573.00,1971.00,1331.00,1211.00


In [11]:
# Categorical / key column value counts
print("\n--- Airline Distribution ---")
print(df_flights['AIRLINE'].value_counts())
print(f"\nUnique Airlines: {df_flights['AIRLINE'].nunique()}")
print(f"Unique Origin Airports: {df_flights['ORIGIN_AIRPORT'].nunique()}")
print(f"Unique Destination Airports: {df_flights['DESTINATION_AIRPORT'].nunique()}")
print(f"\nDate Range: {df_flights['YEAR'].min()}-{df_flights['MONTH'].min():02d}-{df_flights['DAY'].min():02d} to {df_flights['YEAR'].max()}-{df_flights['MONTH'].max():02d}-{df_flights['DAY'].max():02d}")
print(f"\nCancelled Flights: {df_flights['CANCELLED'].sum():,} ({df_flights['CANCELLED'].mean()*100:.2f}%)")
print(f"Diverted Flights: {df_flights['DIVERTED'].sum():,} ({df_flights['DIVERTED'].mean()*100:.2f}%)")


--- Airline Distribution ---
AIRLINE
WN    1261855
DL     875881
AA     725984
OO     588353
EV     571977
UA     515723
MQ     294632
B6     267048
US     198715
AS     172521
NK     117379
F9      90836
HA      76272
VX      61903
Name: count, dtype: int64

Unique Airlines: 14
Unique Origin Airports: 628
Unique Destination Airports: 629

Date Range: 2015-01-01 to 2015-12-31

Cancelled Flights: 89,884 (1.54%)
Diverted Flights: 15,187 (0.26%)


## 1.8 Extraction Summary

| Dataset | Rows | Columns | Key Columns | Missing Values |
|---------|------|---------|-------------|----------------|
| Airlines | 14 | 2 | IATA_CODE, AIRLINE | None |
| Airports | 322 | 7 | IATA_CODE, AIRPORT, CITY, STATE, COUNTRY, LAT, LONG | Possible |
| Flights | ~5.8M | 31 | All delay, timing, airport, airline columns | Yes — delay columns, cancellation reason |

**Key Observations:**
- Flights dataset contains ~5.8 million records across all of 2015
- Missing values are concentrated in delay breakdown columns (expected for non-delayed/cancelled flights)
- CANCELLATION_REASON is missing for non-cancelled flights (expected)
- Time columns (SCHEDULED_DEPARTURE, etc.) are stored as integers in HHMM format — need conversion

**Next Step:** Proceed to `02_cleaning.ipynb` for data cleaning and transformation pipeline.